# Širenje poremećaja u mreži aerodroma
Primjena Sorensen–Zhou / Hammarlingova algoritma na dataset
[2015 Flight Delays and Cancellations (Kaggle, US DOT)](https://www.kaggle.com/datasets/usdot/flight-delays).

**Pitanje:** kako mjesto nastanka poremećaja i brzina oporavka utječu na
varijabilnost cijele mreže? Stvarni letovi određuju veze, a dinamiku zadajemo
pojednostavljenim difuzijskim modelom. Ne procjenjujemo niti prognoziramo stvarna kašnjenja.

Pokrenite ćelije redom u kernelu **Julia Hammarling** iz ovog projekta.
Izvor bilježnice je `flight_delays_lyapunov.jl`; pomoćne funkcije su u `flight_network.jl`.


In [1]:
using LinearAlgebra
using BenchmarkTools
import MatrixEquations

project_root = isfile(joinpath(pwd(), "flight_network.jl")) ? pwd() : @__DIR__
include(joinpath(project_root, "flight_network.jl"))
using .FlightNetwork

module AirportLyapunov
    include(joinpath(@__DIR__, "sorensen_zhou.jl"))
end
using .AirportLyapunov: lyapunov_factor, relative_residual


(real_residual = 4.678790580762225e-17, complex_residual = 6.587146713995547e-17, relative_solution_error = 1.1101076597580674e-16)

Test Summary: | Pass  Total  Time
Sorensen–Zhou |  134    134  9.0s


## 1. Podaci i parametri
Preuzmite i raspakirajte `flights.csv` i `airports.csv` u `data/flight-delays/`.
Alternativno, uključite `download_data`: preuzimanje cijele Kaggle ZIP arhive
može potrajati i zauzima nekoliko stotina MB; lokalne CSV datoteke koriste se ponovno.
Ako javni API traži prijavu ili odbije zahtjev, preuzmite ZIP ručno preko poveznice iznad.
`airlines.csv` nije potreban. Podaci se ne dodaju u Git.

Početna ćelija učitava izvornu implementaciju te pokreće njezin primjer i provjere.
Ovdje ne kopiramo algoritam, pa promjene u `sorensen_zhou.jl` vrijede i za ovu analizu.


In [2]:
data_directory = get(ENV, "FLIGHT_DELAYS_DATA", joinpath(project_root, "data", "flight-delays"))
download_data = false
month = 1
number_of_airports = 50
alpha = 1.0
gamma = 0.2
sigma = 1.0
background_noise = 0.01

ensure_dataset(data_directory; download=download_data)


"/home/carjin/projects/hammarlingov-algoritam/data/flight-delays"

## 2. Čišćenje i agregiranje
`flights.csv` čitamo redak po redak i zadržavamo samo odabrani mjesec.
Izbacujemo otkazane letove, nepoznate/nedostajuće oznake i letove sa istim polazištem i odredištem.
Prihvaćamo samo aerodrome s valjanim koordinatama iz `airports.csv`.
Neke oznake aerodroma u datasetu su numeričke; ne nagađamo njihovo preslikavanje,
nego broj odbačenih redaka izričito prikazujemo. To je posebno važno pri promjeni mjeseca.

Rang aerodroma određuje ukupan broj dolazaka i odlazaka u cijelom prihvaćenom
mjesečnom uzorku. Zatim zadržavamo samo veze između prvih `number_of_airports` čvorova.


In [3]:
airports = load_airports(joinpath(data_directory, "airports.csv"))
route_data = load_routes(joinpath(data_directory, "flights.csv"), airports; month)
network = build_network(route_data.routes, airports; n=number_of_airports)
display(result_table([
    (mjera="Mjesec", vrijednost=month),
    (mjera="Redaka u mjesecu", vrijednost=route_data.selected),
    (mjera="Otkazani letovi", vrijednost=route_data.cancelled),
    (mjera="Nevaljani ili nepoznati podaci", vrijednost=route_data.invalid),
    (mjera="Prihvaćeni letovi prije izbora čvorova", vrijednost=route_data.accepted),
    (mjera="Aerodroma u modelu", vrijednost=length(network.codes)),
    (mjera="Letova unutar odabrane mreže", vrijednost=Int(sum(network.counts))),
]))


mjera,vrijednost
Mjesec,1
Redaka u mjesecu,469968
Otkazani letovi,11982
Nevaljani ili nepoznati podaci,557
Prihvaćeni letovi prije izbora čvorova,457429
Aerodroma u modelu,50
Letova unutar odabrane mreže,284980


## 3. Od mreže do stabilnog sustava
Neka je $C_{ij}$ broj letova iz aerodroma $i$ u aerodrom $j$. Definiramo

$$\widetilde W=(C+C^\mathsf T)/2,\quad
s=\max_i\sum_j\widetilde W_{ij},\quad W=\widetilde W/s,$$
$$D=\operatorname{diag}(W\mathbf1),\quad L=D-W,\quad A=-\alpha L-\gamma I.$$

Simetrizacija zanemaruje smjer letova. Jedinstveni faktor $s$ zadržava relativne
težine veza i daje usporedivu skalu za $\alpha$ i $\gamma$ unutar ovog eksperimenta.
Za simetričnu nenegativnu mrežu $L\succeq0$, pa vrijedi
$\lambda_i(A)=-\alpha\lambda_i(L)-\gamma<0$.
Član $-\gamma I$ nužan je: sam $-L$ ima nultu svojstvenu vrijednost.

Vrijeme je u proizvoljnim jedinicama modela. Parametri nisu procijenjeni iz vremena
letova ili kašnjenja. Izdvajanje podmreže zanemaruje sve veze s ostalim aerodromima.


In [4]:
A = system_matrix(network.L; alpha, gamma)
source_airports = network.codes[1:2]
B = disturbance_matrix(network.codes, source_airports; sigma, background=background_noise)
display((sources=source_airports, size_A=size(A), size_B=size(B),
         largest_eigenvalue=eigmax(Symmetric(A)),
         weight_scale=network.weight_scale))


(sources = ["ATL", "DFW"], size_A = (50, 50), size_B = (50, 52), largest_eigenvalue = -0.20000000000000004, weight_scale = 16424.5)

## 4. Stacionarna kovarijanca i Gramian
Promatramo linearni stohastički sustav
$$dx=A x\,dt+B\,d\mathcal W_t.$$
$x_i$ je apstraktno odstupanje od ravnoteže u aerodromu $i$, a $\mathcal W_t$
sadrži neovisne standardne Wienerove procese. Za stabilan $A$ kovarijanca zadovoljava
$$AX+XA^\mathsf T+BB^\mathsf T=0.$$
To je ujedno Gramian upravljivosti determinističkog sustava $\dot x=Ax+Bu$.
[Referenca za Gramian i Lyapunovljevu jednadžbu](https://www.mathworks.com/help/control/ref/statespacemodel.gram.html).

Svaki odabrani izvor daje stupac $\sigma e_i$ u $B$. Dodajemo i slabu nezavisnu
pozadinsku pobudu $\eta I$, s `η = background_noise`, tako da
$$BB^\mathsf T=\sigma^2\sum_{i\in S}e_ie_i^\mathsf T+\eta^2 I.$$
To izbjegava singularnu kovarijancu kod nedosegljivih modova. Pozadinska pobuda
jednaka je u svim usporedbama; može se isključiti postavljanjem vrijednosti na nulu.
Tada singularnost ili loša uvjetovanost mogu otežati faktorizaciju.

$X_{ii}$ mjeri modeliranu varijancu, $X_{ij}$ kovarijancu, a $\operatorname{tr}(X)$
ukupnu varijabilnost mreže. Te vrijednosti nemaju jedinicu minuta stvarnog kašnjenja.


In [5]:
U_real = lyapunov_factor(A, B)
U_complex = lyapunov_factor(A, B; arithmetic=:complex)
U_reference = MatrixEquations.plyapc(A, B)
X = Matrix(U_real * U_real')
X_reference = U_reference * U_reference'
solver_results = [
    (solver=name, residual=relative_residual(A, B, U),
     relative_error=norm(U * U' - X_reference) / norm(X_reference))
    for (name, U) in [("Sorensen–Zhou real", U_real),
                      ("Sorensen–Zhou complex", U_complex),
                      ("MatrixEquations.plyapc", U_reference)]
]
display(result_table(solver_results))
@assert maximum(row.residual for row in solver_results) < 1e-10
@assert maximum(row.relative_error for row in solver_results) < 1e-8


solver,residual,relative_error
Sorensen–Zhou real,1.0744737733394898e-15,8.79612215333778e-15
Sorensen–Zhou complex,7.701133509507483e-16,6.201143452088291e-15
MatrixEquations.plyapc,6.799573052119504e-16,0.0


## 5. Gdje su poremećaji najizraženiji?
Karta prikazuje geografski raspored bez kartografske podloge; debljina veze ovisi
o broju letova. Tamniji i veći čvor znači veću varijancu. Prelaskom miša preko
čvora dobivaju se naziv i vrijednost. Skala boje relativna je tom prikazu.
Toplinska karta pokazuje apsolutne kovarijance za prvih 20 aerodroma po prometu,
dok tablica rangira sve čvorove prema varijanci. Grafike su SVG, bez dodatnih servisa.


In [6]:
display(airport_map(network, diag(X)))
display(covariance_heatmap(network.codes, X))
ranking = sortperm(diag(X); rev=true)
display(result_table([
    (airport=network.codes[i], name=network.airports[i].name,
     monthly_traffic=network.traffic[i], variance=round(X[i, i]; sigdigits=5))
    for i in ranking[1:min(10, length(ranking))]
]))


HTML{String}("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 920 500' style='max-width:100%;background:#fff;color:#111;font-family:sans-serif'><text x='20' y='25'>Mreža aerodroma: tamnije / veće = veća modelirana varijanca</text><line x1='695.6343030037451' y1='245.63985803300707' x2='583.141396627393' y2='255.64866547441602' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.41845109440165607'/><line x1='695.6343030037451' y1='245.63985803300707' x2='664.6122359106089' y2='133.529491657233' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.38831319066029407'/><line x1='583.141396627393' y1='255.64866547441602' x2='664.6122359106089' y2='133.529491657233' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.4213735577947578'/><line x1='695.6343030037451' y1='245.63985803300707' x2='392.49732758431253' y2='241.57847275174112' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3690431976620293'/><line x1='583.141396627393' y1='255.64866547441602' x2='392.49732758431253' y2='241.57847275174112' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.41269749459648697'/><line x1='664.6122359106089' y1='133.529491657233' x2='392.49732758431253' y2='241.57847275174112' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.413976072330969'/><line x1='695.6343030037451' y1='245.63985803300707' x2='515.0779009444893' y2='162.04643917384476' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.36968248652927027'/><line x1='583.141396627393' y1='255.64866547441602' x2='515.0779009444893' y2='162.04643917384476' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.398907120460288'/><line x1='664.6122359106089' y1='133.529491657233' x2='515.0779009444893' y2='162.04643917384476' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.38292489877926267'/><line x1='392.49732758431253' y1='241.57847275174112' x2='515.0779009444893' y2='162.04643917384476' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3878565557551219'/><line x1='695.6343030037451' y1='245.63985803300707' x2='598.2841818448056' y2='294.8439238650982' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.361554385217206'/><line x1='583.141396627393' y1='255.64866547441602' x2='598.2841818448056' y2='294.8439238650982' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.37954580048098874'/><line x1='664.6122359106089' y1='133.529491657233' x2='598.2841818448056' y2='294.8439238650982' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3619196931413437'/><line x1='392.49732758431253' y1='241.57847275174112' x2='598.2841818448056' y2='294.8439238650982' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.362193674084447'/><line x1='515.0779009444893' y1='162.04643917384476' x2='598.2841818448056' y2='294.8439238650982' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.38137234010167737'/><line x1='695.6343030037451' y1='245.63985803300707' x2='449.59018100190406' y2='248.41291974094054' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.3598191725775518'/><line x1='583.141396627393' y1='255.64866547441602' x2='449.59018100190406' y2='248.41291974094054' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.37379220067581964'/><line x1='664.6122359106089' y1='133.529491657233' x2='449.59018100190406' y2='248.41291974094054' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.37351821973271637'/><line x1='392.49732758431253' y1='241.57847275174112' x2='449.59018100190406' y2='248.41291974094054' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.422469481567171'/><line x1='515.0779009444893' y1='162.04643917384476' x2='449.59018100190406' y2='248.41291974094054' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.41991232609820694'/><line x1='598.2841818448056' y1='294.8439238650982' x2='449.59018100190406' y2='248.41291974094054' stroke='#94a3b8' stroke-opacity='0.25' stroke-width='0.35488751560169257'/><line x1='695.6343030037451' y1='245.63985803300707' x2='357.11078628964265' y2='192.152717931606' stroke='#94a3b8' s

HTML{String}("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 560 530' style='max-width:650px;width:100%;background:#fff;font-family:sans-serif'><text x='15' y='18'>|Xᵢⱼ| za prvih 20 aerodroma; tamnije = veće</text><rect x='90' y='65' width='22' height='22' fill='rgb(70,145,183)'><title>ATL / ATL: 0.4244179406574428</title></rect><rect x='90' y='87' width='22' height='22' fill='rgb(226,239,251)'><title>DFW / ATL: 0.024019944589045848</title></rect><rect x='90' y='109' width='22' height='22' fill='rgb(231,243,253)'><title>ORD / ATL: 0.00991094944067137</title></rect><rect x='90' y='131' width='22' height='22' fill='rgb(232,243,254)'><title>LAX / ATL: 0.00780458279410999</title></rect><rect x='90' y='153' width='22' height='22' fill='rgb(232,243,254)'><title>DEN / ATL: 0.008263396086524434</title></rect><rect x='90' y='175' width='22' height='22' fill='rgb(232,243,254)'><title>IAH / ATL: 0.00786603755619913</title></rect><rect x='90' y='197' width='22' height='22' fill='rgb(232,243,254)'><title>PHX / ATL: 0.007277105464664455</title></rect><rect x='90' y='219' width='22' height='22' fill='rgb(232,243,254)'><title>SFO / ATL: 0.0064944337804379295</title></rect><rect x='90' y='241' width='22' height='22' fill='rgb(232,243,254)'><title>LAS / ATL: 0.006917776996217456</title></rect><rect x='90' y='263' width='22' height='22' fill='rgb(230,242,253)'><title>MCO / ATL: 0.012939025137083775</title></rect><rect x='90' y='285' width='22' height='22' fill='rgb(231,243,253)'><title>DTW / ATL: 0.009044518915722189</title></rect><rect x='90' y='307' width='22' height='22' fill='rgb(231,242,253)'><title>CLT / ATL: 0.011409463776169582</title></rect><rect x='90' y='329' width='22' height='22' fill='rgb(232,243,254)'><title>MSP / ATL: 0.008790242908676549</title></rect><rect x='90' y='351' width='22' height='22' fill='rgb(230,242,253)'><title>LGA / ATL: 0.012730056639043337</title></rect><rect x='90' y='373' width='22' height='22' fill='rgb(233,244,254)'><title>SLC / ATL: 0.005534946760679752</title></rect><rect x='90' y='395' width='22' height='22' fill='rgb(233,244,254)'><title>SEA / ATL: 0.005025272682690147</title></rect><rect x='90' y='417' width='22' height='22' fill='rgb(232,243,254)'><title>BOS / ATL: 0.008231296535133001</title></rect><rect x='90' y='439' width='22' height='22' fill='rgb(232,243,254)'><title>EWR / ATL: 0.008753280142128828</title></rect><rect x='90' y='461' width='22' height='22' fill='rgb(233,244,254)'><title>JFK / ATL: 0.004550602343725411</title></rect><rect x='90' y='483' width='22' height='22' fill='rgb(230,242,253)'><title>FLL / ATL: 0.01256537960317719</title></rect><rect x='112' y='65' width='22' height='22' fill='rgb(226,239,251)'><title>ATL / DFW: 0.024019944589045848</title></rect><rect x='112' y='87' width='22' height='22' fill='rgb(30,120,165)'><title>DFW / DFW: 0.5288918281942427</title></rect><rect x='112' y='109' width='22' height='22' fill='rgb(229,241,252)'><title>ORD / DFW: 0.015697457296124616</title></rect><rect x='112' y='131' width='22' height='22' fill='rgb(230,242,253)'><title>LAX / DFW: 0.014181520192906315</title></rect><rect x='112' y='153' width='22' height='22' fill='rgb(230,242,253)'><title>DEN / DFW: 0.013700650264191816</title></rect><rect x='112' y='175' width='22' height='22' fill='rgb(230,242,253)'><title>IAH / DFW: 0.012319832382521184</title></rect><rect x='112' y='197' width='22' height='22' fill='rgb(231,242,253)'><title>PHX / DFW: 0.01131335339531273</title></rect><rect x='112' y='219' width='22' height='22' fill='rgb(230,242,253)'><title>SFO / DFW: 0.01199278114434063</title></rect><rect x='112' y='241' width='22' height='22' fill='rgb(230,242,253)'><title>LAS / DFW: 0.011785014565716494</title></rect><rect x='112' y='263' width='22' height='22' fill='rgb(230,242,253)'><title>MCO / DFW: 0.01216770353630966</title></rect><rect x='112' y='285' width='22' height='22' fill='rgb(232,243,254)'><title>DTW / DFW: 0.008269494262113692</title></rect><rect x='112' y='307' width='22' h

airport,name,monthly_traffic,variance
DFW,Dallas/Fort Worth International Airport,45232,0.52889
ATL,Hartsfield-Jackson Atlanta International Airport,58124,0.42442
MCO,Orlando International Airport,19781,0.002389
LGA,LaGuardia Airport (Marine Air Terminal),17299,0.0023753
ORD,Chicago O'Hare International Airport,44663,0.0022365
CLT,Charlotte Douglas International Airport,18407,0.0021581
FLL,Fort Lauderdale-Hollywood International Airport,14007,0.0021346
PHL,Philadelphia International Airport,10941,0.0020957
TPA,Tampa International Airport,11733,0.0019453
DCA,Ronald Reagan Washington National Airport,12728,0.001924


## 6. Jednaka pobuda u prometnom i manje prometnom čvoru
Uspoređujemo prvi i posljednji aerodrom prema prometu **unutar odabranog skupa**.
Svaki scenarij ima jedan lokalni izvor iste jačine i jednaku pozadinsku pobudu.
Posebno izdvajamo varijancu izvan izvornog čvora: ona opisuje koliko se učinak
proširio na ostatak mreže. Velik lokalni poremećaj ne mora značiti velik utjecaj drugdje.


In [7]:
source_comparison = map([first(network.codes), last(network.codes)]) do source
    Bs = disturbance_matrix(network.codes, [source]; sigma, background=background_noise)
    Us = lyapunov_factor(A, Bs)
    Xs = Us * Us'
    index = findfirst(==(source), network.codes)
    (source=source, local_variance=Xs[index, index], total_variance=tr(Xs),
     variance_elsewhere=tr(Xs) - Xs[index, index], residual=relative_residual(A, Bs, Us))
end
display(result_table(source_comparison))


source,local_variance,total_variance,variance_elsewhere,residual
ATL,0.42335712438264456,0.4612133616814771,0.03785623729883253,7.555699495645102e-16
PIT,1.5510447470746147,1.5625274302907037,0.011482683216089029,2.1598354180179986e-16


## 7. Utjecaj brzine oporavka
Mijenjamo samo $\gamma$, uz istu mrežu, normalizaciju, $\alpha$ i $B$.
Manji $\gamma$ znači sporiji oporavak. Za ovaj simetrični model
$$\operatorname{tr}(X)=\tfrac12\operatorname{tr}((\alpha L+\gamma I)^{-1}BB^\mathsf T),$$
pa ukupna varijanca pada s porastom $\gamma$. To ujedno daje nezavisnu provjeru
očekivanog ponašanja rezultata, uz provjeru reziduala.


In [8]:
recovery_results = map([0.02, 0.05, 0.1, 0.2, 0.5, 1.0]) do recovery_rate
    Ag = system_matrix(network.L; alpha, gamma=recovery_rate)
    Ug = lyapunov_factor(Ag, B)
    Xg = Ug * Ug'
    trace_check = tr((-Ag) \ (B * B')) / 2
    @assert isapprox(tr(Xg), trace_check; rtol=1e-8)
    (gamma=recovery_rate, total_variance=tr(Xg),
     max_variance=maximum(real, diag(Xg)), residual=relative_residual(Ag, B, Ug))
end
@assert all(diff(getproperty.(recovery_results, :total_variance)) .< 0)
display(result_table(recovery_results))
display(recovery_plot(recovery_results))


gamma,total_variance,max_variance,residual
0.02,2.1212988448355254,0.6754174473429563,7.749829012485941e-16
0.05,1.4787367556848736,0.6374630124098348,9.360217890300942e-16
0.1,1.2178188187217798,0.5948602353712512,7.872048961323943e-16
0.2,1.0175178988199038,0.5288918281942427,1.0744737733394898e-15
0.5,0.7525227669146192,0.3996741207599268,6.74815708893342e-16
1.0,0.5413088963259897,0.28497220367244724,1.6122410808984774e-15


HTML{String}("<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 740 390' style='max-width:100%;background:#fff;font-family:sans-serif'><text x='30' y='25'>Ukupna stacionarna varijanca tr(X) i brzina oporavka γ</text><path d='M70 50 V320 H700' fill='none' stroke='#475569'/><text x='60' y='324' text-anchor='end'>0</text><text x='60' y='64' text-anchor='end' font-size='11'>2.12</text><polyline points='70.0,60.0 212.87680457596448,138.75653333140946 320.95893485865486,170.73631014388602 429.0410651413452,195.2864763315858 571.9178697173097,227.765980321848 680.0,253.65370778030592' fill='none' stroke='#155e75' stroke-width='2'/><circle cx='70.0' cy='60.0' r='4' fill='#155e75'><title>γ=0.02, tr(X)=2.1212988448355254</title></circle><text x='70.0' y='340' text-anchor='middle' font-size='11'>0.02</text><circle cx='212.87680457596448' cy='138.75653333140946' r='4' fill='#155e75'><title>γ=0.05, tr(X)=1.4787367556848736</title></circle><text x='212.87680457596448' y='340' text-anchor='middle' font-size='11'>0.05</text><circle cx='320.95893485865486' cy='170.73631014388602' r='4' fill='#155e75'><title>γ=0.1, tr(X)=1.2178188187217798</title></circle><text x='320.95893485865486' y='340' text-anchor='middle' font-size='11'>0.1</text><circle cx='429.0410651413452' cy='195.2864763315858' r='4' fill='#155e75'><title>γ=0.2, tr(X)=1.0175178988199038</title></circle><text x='429.0410651413452' y='340' text-anchor='middle' font-size='11'>0.2</text><circle cx='571.9178697173097' cy='227.765980321848' r='4' fill='#155e75'><title>γ=0.5, tr(X)=0.7525227669146192</title></circle><text x='571.9178697173097' y='340' text-anchor='middle' font-size='11'>0.5</text><circle cx='680.0' cy='253.65370778030592' r='4' fill='#155e75'><title>γ=1.0, tr(X)=0.5413088963259897</title></circle><text x='680.0' y='340' text-anchor='middle' font-size='11'>1.0</text><text x='300' y='375' font-size='13'>γ (logaritamska os)</text></svg>")

## 8. Vrijeme izvođenja
Pokretanje je opcionalno. BenchmarkTools zagrijava funkcije i mjeri ponovljene
pozive na istim matricama; prikazujemo medijan. Učitavanje CSV-a i crtanje nisu
uključeni. Sve tri metode računaju faktor, a Schurova dekompozicija ulazi u mjerenje.
Simetrični $A$ ima realan spektar, pa ovaj primjer ne ispituje realne Schurove
blokove veličine 2. Takvi testovi ostaju u izvornoj bilježnici.


In [9]:
run_timing = false
if run_timing
    timings = [
        (solver="Sorensen–Zhou real", trial=@benchmark(lyapunov_factor($A, $B), samples=10, evals=1)),
        (solver="Sorensen–Zhou complex", trial=@benchmark(lyapunov_factor($A, $B; arithmetic=:complex), samples=10, evals=1)),
        (solver="MatrixEquations.plyapc", trial=@benchmark(MatrixEquations.plyapc($A, $B), samples=10, evals=1)),
    ]
    display(result_table([
        (solver=row.solver, median_ms=median(row.trial).time / 1e6,
         memory_bytes=median(row.trial).memory) for row in timings
    ]))
end


## Ograničenja i mogući nastavci
- Mreža prikazuje broj letova, bez kapaciteta zrakoplova, presjedanja i rasporeda.
- Simetrizacija i difuzija su pretpostavke modela, a ne empirijski utvrđena dinamika.
- Pri usporedbi mjeseci normalizacija se ponovno računa; za čistu usporedbu
  sezonskih promjena trebalo bi zadržati iste čvorove i isti faktor normalizacije.
- Pozadinski šum osigurava pozitivno definitno rješenje, ali mijenja model.
- Nakon promjene parametara pokrenite sve ćelije od početka da prikazi budu usklađeni.

Daljnji korak bio bi procijeniti dinamiku iz vremenskih nizova kašnjenja i validirati
model na izdvojenim podacima. Diskretni model zahtijeva diskretnu Lyapunovljevu jednadžbu,
koju ovdje implementirani kontinuirani solver ne rješava izravno.
